# 01 — Data Ingestion

Load the MedQuAD dataset, chunk documents, and save processed parquet files.

In [ ]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.ingestion.medical_loader import MedQuADLoader
from src.ingestion.chunker import MedicalChunker

## 1. Load data

In [ ]:
loader = MedQuADLoader()
documents, eval_pairs, test_pairs = loader.load()
print(f"Documents: {len(documents)}, Eval pairs: {len(eval_pairs)}, Test pairs: {len(test_pairs)}")

## 2. Chunk documents

In [ ]:
chunker = MedicalChunker(max_chunk_size=1000)
all_chunks = []
for doc in documents:
    chunks = chunker.chunk_medquad(
        question=doc["question"],
        answer=doc["text"],
        metadata={"qtype": doc["qtype"]},
    )
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")

## 3. Save chunks

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

chunks_df = pd.DataFrame(all_chunks)
chunks_df.to_parquet(output_dir / "medical_chunks.parquet", index=False)
print(f"Saved medical_chunks.parquet: {len(chunks_df)} rows")

## 4. Save eval queries

In [ ]:
eval_df = pd.DataFrame(eval_pairs)
eval_df.to_parquet(output_dir / "eval_queries.parquet", index=False)
print(f"Saved eval_queries.parquet: {len(eval_df)} rows")

## 5. Save test queries

In [ ]:
test_df = pd.DataFrame(test_pairs)
test_df.to_parquet(output_dir / "test_queries.parquet", index=False)
print(f"Saved test_queries.parquet: {len(test_df)} rows")

## 6. Summary statistics

In [ ]:
print("--- Summary ---")
print(f"Chunks by qtype:\n{chunks_df['qtype'].value_counts()}")

single = chunks_df[chunks_df['total_chunks'] == 1]
multi = chunks_df[chunks_df['total_chunks'] > 1]
print(f"\nSingle-chunk answers: {len(single)}")
print(f"Multi-chunk answers: {len(multi)} (from {multi['question'].nunique()} Q&A pairs)")

print(f"\nChunk length stats:")
print(chunks_df['text'].str.len().describe())